<a href="https://colab.research.google.com/github/YashThirani21/docagent/blob/main/notebooks/colab_notebook_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# docagent on Google Colab (vLLM), from GitHub

Run top to bottom on a GPU runtime (Runtime → Change runtime type → GPU). The code is cloned from
[github.com/YashThirani21/docagent](https://github.com/YashThirani21/docagent). **No setup or tokens needed.**

Where the run outputs (eval results, a snapshot of the traces, logs, drawing outputs and the figure descriptions)
go is picked in cell 1 by `PERSIST`. The report index is not saved: ingest rebuilds it in ~30 s from the figure
descriptions committed in the repo, so every session uses the same index.
- `"none"` (default): the run outputs are downloaded to your computer as a zip. Nothing leaves Colab otherwise.
- `"drive"`: your own `MyDrive/ColabProjects/run_out/` (asks for Drive access once).
- `"hf"`: a private Hugging Face dataset repo, `<your HF username>/docagent-store`, in the account of whoever runs the
  notebook. Every save is a commit. Needs a Colab secret `HF_TOKEN` (🔑 icon in the left bar) with write access.
- `"auto"`: `"hf"` if this Colab account has an `HF_TOKEN` secret, otherwise `"none"`.

## 1. Settings

In [1]:
import os, time
from pathlib import Path

REPO_URL = "https://github.com/YashThirani21/docagent.git"
REF = "main"
PROJECT = Path("/content/senior-data-scientist")

PERSIST = "none"                # where run outputs go: "none" (zip download), "drive", "hf" or "auto" (hf if HF_TOKEN)
HF_DATASET = None               # for "hf" only: <token owner>/docagent-store, created private on the first save
DRIVE_DIR = Path("/content/drive/MyDrive/ColabProjects")

RUN_ID = time.strftime("%Y%m%d-%H%M%S")             # run outputs are saved under runs/<RUN_ID>/

if PERSIST in ("auto", "hf"):
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        from huggingface_hub import whoami
        # Each person saves to their own account: anyone else's token can't write to the maintainer's repo
        HF_DATASET = HF_DATASET or f"{whoami()['name']}/docagent-store"
        PERSIST = "hf"
    except Exception as e:      # no such secret, notebook access not granted, or the token is invalid
        if PERSIST == "hf":
            raise RuntimeError(
                f"No usable Colab secret HF_TOKEN ({type(e).__name__}). Add a token with Write access in "
                "Colab's 🔑 Secrets panel as HF_TOKEN with 'Notebook access' on, or set PERSIST = \"none\"."
            ) from None
        PERSIST = "none"
if PERSIST == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
elif PERSIST not in ("hf", "none"):
    raise ValueError(f"PERSIST must be 'auto', 'none', 'drive' or 'hf', not {PERSIST!r}")
print(f"persist: {PERSIST}" + (f" ({HF_DATASET})" if PERSIST == "hf" else "") + f", run: {RUN_ID}")

persist: none, run: 20261021-172816


## 2. Clone the project

In [2]:
import sys
if "chat" in sys.modules:
    # The running chat app keeps store/traces.db open: deleting it now sends its traces to a deleted file
    raise RuntimeError("The chat app is running from this project. Runtime → Restart session, then run from cell 1.")
!rm -rf {PROJECT}
!git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}
!git checkout -q {REF} && git log -1 --format="running %h  %s  (%cd)" --date=short

/content/senior-data-scientist
running 7928640  Created using Colab  (2026-10-21)


## 3. Save helper

`push_run()` saves a run's outputs wherever `PERSIST` points (a zip download when it is `"none"`), so switching
`PERSIST` changes nothing else in the notebook. Hugging Face uploads run through the `hf` CLI with
`HF_HUB_OFFLINE=0`, because the docagent config puts the kernel in offline mode once it is loaded
(`privacy.offline`).

In [3]:
import shutil, sqlite3, subprocess

# figure_descriptions.json: new descriptions (a new model or prompt) come back with the run, to commit
RUN_ITEMS = ["outputs", "eval/results", "store/logs", "store/cache/figure_descriptions.json",
             "vllm.log"]                                  # + store/traces.db, snapshotted below
TRACES_DB = Path("store/traces.db")

def _hf(*args):
    env = {**os.environ, "HF_HUB_OFFLINE": "0"}
    return subprocess.run(["hf", *args, "--repo-type", "dataset"], env=env, capture_output=True, text=True)

def push_run():
    """Save outputs, eval results, traces and logs under runs/<RUN_ID>/."""
    staging = Path("/content/run_out") / RUN_ID
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    for item in RUN_ITEMS:
        p = PROJECT / item
        if p.is_dir():
            shutil.copytree(p, staging / p.name)
        elif p.is_file():
            shutil.copy2(p, staging / p.name)
        else:
            print(f"skipped (not found): {item}")
    if (PROJECT / TRACES_DB).is_file():
        # A copy of the file alone misses what is still in traces.db-wal; the backup API includes it
        with sqlite3.connect(PROJECT / TRACES_DB) as src, sqlite3.connect(staging / TRACES_DB.name) as dst:
            src.backup(dst)
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(staging), f"runs/{RUN_ID}", "--private",
                "--commit-message", f"run {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        print(f"Run saved: hf.co/datasets/{HF_DATASET}/tree/main/runs/{RUN_ID}")
    elif PERSIST == "none":
        from google.colab import files
        zip_path = shutil.make_archive(f"/content/docagent-run-{RUN_ID}", "zip", staging)
        files.download(zip_path)
        print(f"Run downloaded to your computer: {Path(zip_path).name}")
    else:
        shutil.copytree(staging, DRIVE_DIR / "run_out" / RUN_ID, dirs_exist_ok=True)
        print(f"Run saved: {DRIVE_DIR / 'run_out' / RUN_ID}")

## 4. Install vLLM and the docagent package

In [5]:
%cd /content/senior-data-scientist

!pip install -q vllm==0.31.0 -e . -c requirements-gpu.txt

import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import docagent, vllm; print('docagent and vllm', vllm.__version__, 'installed')"],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Install failed; rerun this cell without -q to see pip's error.\n" + r.stderr[-1500:])
print(r.stdout.strip())

/content/senior-data-scientist
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

## 5. Check the GPU

In [6]:
import shutil, subprocess
# vLLM and the bge-m3 embedder (device: cuda in the colab configs) both need a GPU; stop before cells 7-10 fail
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi"], capture_output=True).returncode:
    raise RuntimeError("No GPU in this session. Runtime → Change runtime type → T4 GPU (or L4/A100), "
                       "then Runtime → Run all. If no GPU is offered, Colab's free GPU quota is used up for now.")
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA L4, 23034 MiB, 0 MiB


## 6. Pick the config and start the matching vLLM server

In [7]:
import os, yaml
%cd /content/senior-data-scientist

# Pick the agent version here; the vLLM server is started to match it.
CONFIG = "configs/colab_vllm_awq.yaml"   # or "configs/colab_vllm.yaml"
# CONFIG = "configs/colab_vllm.yaml"
MAX_LEN = 16384                          # vLLM --max-model-len, as in the configs (results up to 2026-10-19 ran at 8192)
GPU_UTIL = 0.90                          # share of GPU memory for vLLM; the rest is for bge-m3 on cuda

os.environ["DOCAGENT_CONFIG"] = CONFIG   # CLI, chat app and Agent() all read this
SERVED = yaml.safe_load(open(CONFIG))["llm"]["model"]   # vLLM must serve exactly the name the config asks for

print(SERVED)
if SERVED.endswith("-awq"):
    # AWQ 4-bit weights in fp16: fits a T4
    print(f"{CONFIG}: serving AWQ as {SERVED}")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} MODEL=Qwen/Qwen2.5-VL-7B-Instruct-AWQ QUANT=awq DTYPE=half SERVED={SERVED} bash scripts/start_vllm.sh
else:
    # start_vllm.sh picks the weights for the GPU: bf16 on L4/A100, AWQ on a T4
    print(f"{CONFIG}: serving as {SERVED}, weights chosen by GPU")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} SERVED={SERVED} bash scripts/start_vllm.sh

/content/senior-data-scientist
qwen2.5-vl-7b-awq
configs/colab_vllm_awq.yaml: serving AWQ as qwen2.5-vl-7b-awq
GPU 23034 MB, compute 89: serving Qwen/Qwen2.5-VL-7B-Instruct-AWQ as qwen2.5-vl-7b-awq (quant=awq dtype=half util=0.9 images=4)
vLLM starting (pid 3893), log: vllm.log
vLLM ready


## 7. Check the server responds

In [8]:
!curl -s http://127.0.0.1:8000/v1/models | head -c 200

{"object":"list","data":[{"id":"qwen2.5-vl-7b-awq","object":"model","created":1792604226,"owned_by":"vllm","root":"Qwen/Qwen2.5-VL-7B-Instruct-AWQ","parent":null,"max_model_len":16384,"permission":[{"

## 8. Download the embedding model (bge-m3)

In [9]:
!python scripts/download_models.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0% 0/30 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/191 [00:00<?, ?B/s]           

Fetching 30 files:   3% 1/30 [00:00<00:06,  4.30it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 191/2.10M [00:00<42:19, 827B/s]
Reconstructing (incomplete total...):   0% 191/2.12M [00:00<42:38, 827B/s]
Reconstructing (incomplete total...):   1% 16.0k/2.12M [00:00<42:21, 827B/s]
Reconstructing (incomplete total...):   1% 17.6k/2.12M [00:00<42:19, 827B/s]
Reconstructing (incomplete total...):   1% 17.8k/2.12M [00:00<42:20, 827B/s]
Reconstructing (incomplete total...):   1% 18.4k/2.13M [00:00<42:27, 827B/s]
Reconstructing (incomplete total...):   1% 24.6k/2.26M [00:00<44:59, 827B/s]
Reconstructing (incomplete total...):   6% 156k/2.74M [00:00<52:06,

## 9. Ingest the reports

In [10]:
!python -m docagent.cli ingest data/reports
!python -m docagent.cli status

Loading weights: 100% 391/391 [00:00<00:00, 2669.20it/s]
pre tokenize: 100% 1/1 [00:00<00:00, 442.95it/s]
Inference Embeddings: 100% 1/1 [00:00<00:00, 46.46it/s]
/content/senior-data-scientist/src/docagent/store.py:46: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(self.collection, field, qm.PayloadSchemaType.KEYWORD)
pre tokenize: 100% 2/2 [00:00<00:00, 260.18it/s]
Inference Embeddings: 100% 2/2 [00:00<00:00,  3.40it/s]
pre tokenize: 100% 4/4 [00:00<00:00, 228.40it/s]
Inference Embeddings: 100% 4/4 [00:00<00:00,  4.12it/s]
ntsb_16-006.pdf: 26 chunks (12 figures)
ntsb_18-061.pdf: 64 chunks (33 figures)
ntsb_24-013.pdf: 6 chunks (5 figures)
collection reports_bge-m3_v1_bf16desc: 96 chunks
  ntsb_16-006: 26
  ntsb_18-061: 64
  ntsb_24-013: 6


## 10. Launch the chat app (public Gradio share link)

In [11]:
import sys
sys.path.insert(0, "/content/senior-data-scientist/src")
sys.path.insert(0, "/content/senior-data-scientist/app")

import chat
from IPython.display import HTML, display

demo = chat.demo

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=True,
    prevent_thread_lock=True,
    inline=False,
)

url = demo.share_url
display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0411ab682dff93d940.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Save run outputs, eval results, traces and logs

In [13]:
push_run()

skipped (not found): outputs


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Run downloaded to your computer: docagent-run-20261021-172816.zip


## 12. Ask a few questions in the notebook

In [14]:
import sys, time
sys.path.insert(0, "/content/senior-data-scientist/src")
from IPython.display import Markdown, display
from docagent.agent import Agent
from docagent.render import render_markdown

questions = [
    "What is the fatigue life of a 2024-T3 spar in cycles?",
    "Why did the left wing main spar fail?",
    "What caused the throttle arm to fracture?",
]
agent = chat.agent if "chat" in globals() else Agent()
batch_id = "questions-" + time.strftime("%Y%m%d-%H%M%S")
for q in questions:
    resp = agent.ask(q, batch_id=batch_id)
    display(Markdown(f"### {q}\n\n" + render_markdown(resp)))
print("traces:", batch_id)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 44.45it/s]


### What is the fatigue life of a 2024-T3 spar in cycles?

The fatigue life of a 2024-T3 spar cannot be determined from the provided information. The reports focus on the examination of the spar after the accident, detailing the presence of fatigue features and the origin of these fatigue cracks. However, they do not provide data on the fatigue life of the material in terms of cycles. The reports mention the hardness and conductivity of the spar, but these properties are not directly related to fatigue life. Additionally, the reports do not provide any information on the cyclic loading history of the spar during its service life.

**Sources:** ntsb_18-061 p.1; ntsb_18-061 p.7; ntsb_18-061 p.4

<sub>request 7cc0e4b1c469 · routes: report_qa (rule)</sub>

### Why did the left wing main spar fail?

The left wing main spar failed due to fatigue cracking originating from the corner radius between the hole LC-1 bore and the main spar lower surface. The crack extended from the forward side of hole LC-1, consistent with fatigue features such as ratchet marks and curving crack arrest lines. The fatigue region was observed to extend up to 0.126 inches forward from the hole surface and up to 0.115 inches upward from the lower surface.

**Sources:** ntsb_18-061 p.7; ntsb_18-061 p.25 (Fig. 19)

<sub>request de1798b1fbfc · routes: report_qa (rule)</sub>

### What caused the throttle arm to fracture?

The throttle arm fractured due to bending overstress.

**Sources:** ntsb_24-013 p.3; ntsb_24-013 p.4 (Fig. 2); ntsb_24-013 p.6 (Fig. 4); ntsb_24-013 p.6 (Fig. 5); ntsb_24-013 p.5 (Fig. 3); ntsb_24-013 p.4 (Fig. 1); ntsb_16-006 p.6 (Fig. 1)

<sub>request 4e955082583c · routes: report_qa (rule)</sub>

traces: questions-20261021-174203


## 13. Spans of one request (the latest by default)

In [15]:
!python -m docagent.cli trace last

request 4e955082583c  succeeded  3.4s  created 2026-10-21T17:42:32.179Z  batch=questions-20261021-174203
  text:     What caused the throttle arm to fracture?
  versions: {"router": "rules+vlm-v1", "drawing": "drawing-geometry-v1", "report_qa": "report-qa-v5", "llm": "qwen2.5-vl-7b-awq"}  config=b9ca2ee192d1
  llm:      1 model calls, tokens 2679 in / 116 out
  outcome:  {"routes": ["report_qa"], "errors": [], "drawings": [], "answer": "The throttle arm fractured due to bending overstress.", "grounded": true, "citations": ["ntsb_24-013 p3", "ntsb_24-013 p4 fig2", "ntsb_24-013 p6 fig4", "ntsb_24-013 p6 fig5", "ntsb_24-013 p5 fig3", "ntsb_24-013 p4 fig1", "ntsb_16-...
spans:
  router  0.00s  {"n_attachments": 0, "route": "report_qa", "tier": "rule"}
  report_qa  3.39s  {"has_text": true, "n_images": 0}
    qa.retrieve  0.08s  {"query": "What caused the throttle arm to fracture?", "retriever": "hybrid-rrf-v3-textfirst-reg-focus-next...
    qa.generate  3.25s  {"n_sources": 11, "n_images":

## 14. Trace of the latest request

---



In [16]:
import sqlite3, pandas as pd

con = sqlite3.connect(PROJECT / "store/traces.db")
REQUEST_ID = pd.read_sql("select request_id from requests order by created_at desc limit 1", con).iloc[0, 0]
# REQUEST_ID = "53b8c2c82def"   # or any id from a chat reply / `docagent requests`
print("request", REQUEST_ID)
pd.read_sql("select * from spans where request_id = ? order by started_at", con, params=(REQUEST_ID,))

request 4e955082583c


,span_id,parent_id,request_id,task_id,name,status,started_at,ended_at,duration_ms,attrs,error
0,2b2fface7466,None,4e955082583c,None,router,succeeded,2026-10-21T17:42:32.179Z,2026-10-21T17:42:32.179Z,0.176230,"{""n_attachments"": 0, ""route"": ""report_qa"", ""ti...",None
1,568f99bef9a3,None,4e955082583c,None,report_qa,succeeded,2026-10-21T17:42:32.180Z,2026-10-21T17:42:35.572Z,3392.237067,"{""has_text"": true, ""n_images"": 0}",None
2,c4433747e298,568f99bef9a3,4e955082583c,None,qa.retrieve,succeeded,2026-10-21T17:42:32.180Z,2026-10-21T17:42:32.258Z,78.214503,"{""query"": ""What caused the throttle arm to fra...",None
3,f6536f9fefe6,568f99bef9a3,4e955082583c,None,qa.generate,succeeded,2026-10-21T17:42:32.325Z,2026-10-21T17:42:35.571Z,3245.790998,"{""n_sources"": 11, ""n_images"": 2, ""n_user_image...",None
4,6f53b28e01a6,f6536f9fefe6,4e955082583c,None,llm.call,succeeded,2026-10-21T17:42:32.443Z,2026-10-21T17:42:35.571Z,3128.042106,"{""model"": ""qwen2.5-vl-7b-awq"", ""n_images"": 2, ...",None
5,23acdb46ecf3,568f99bef9a3,4e955082583c,None,qa.verify_citations,succeeded,2026-10-21T17:42:35.571Z,2026-10-21T17:42:35.572Z,0.736417,"{""model_cited"": [0, 5, 6, 7, 8, 9, 10], ""kept""...",None


## 15. Evaluate the model in memory

The next cell finds the model vLLM is serving and its config, and compares each run with the latest pinned run of the
same model in `eval/results/`. The eval runs inside this kernel, next to the chat app (see the cell).

In [17]:
import json, os, subprocess, sys, urllib.request, yaml
from pathlib import Path

PROJECT = Path("/content/senior-data-scientist")
LABEL = None              # None: <awq|bf16>-<gpu>, e.g. bf16-l4: the tag stored with each run
BASELINES = Path("eval/results")   # pinned earlier runs; the latest one of the same model is compared

# Rerunnable, and safe next to the chat app:
# - self-contained: after "Restart session" the kernel forgets CONFIG and the working directory, but vLLM keeps running
# - the eval runs inside this kernel, not as `!python -m eval`: embedded Qdrant (store/qdrant) allows one process,
#   and once the chat app or step 12 has asked a report question, this kernel holds it. In here the eval reuses
#   that open index and the loaded bge-m3 instead of failing with "already accessed by another instance".
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))          # the project's eval/ package (only docagent is pip-installed)
from eval.__main__ import main as eval_main

try:
    with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as r:
        served = [m["id"] for m in json.load(r)["data"]]
except OSError:
    raise RuntimeError("vLLM is not running: run step 6 first") from None
llm = {c: yaml.safe_load(c.read_text())["llm"] for c in sorted(Path("configs").glob("*.yaml"))}
matches = [c for c, l in llm.items() if l.get("backend") == "vllm" and l["model"] in served]
if len(matches) != 1:
    raise RuntimeError(f"vLLM serves {served}; configs asking for it: {[str(c) for c in matches] or 'none'}")
EVAL_CONFIG = matches[0]                      # the config of the model in memory
MODEL = llm[EVAL_CONFIG]["model"]
if LABEL is None:
    try:                                      # "NVIDIA L4" -> l4, "Tesla T4" -> t4
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.split("\n")[0].split()[-1].lower()
    except (OSError, IndexError):
        gpu = "gpu"
    LABEL = ("awq" if MODEL.endswith("-awq") else "bf16") + "-" + gpu
print(f"evaluating {EVAL_CONFIG} ({MODEL}) as {LABEL!r}")

def baseline(suite):
    """Latest pinned run of this suite with the model in memory, on all cases and with --fresh (as run_eval runs)."""
    found = []
    for p in BASELINES.rglob(f"*_{suite}_*.json"):
        try:
            run = json.loads(p.read_text())["run"]
        except (OSError, ValueError, KeyError, TypeError):
            continue
        if run.get("suite") == suite and run.get("model") == MODEL and run.get("split") == "all" and run.get("fresh"):
            found.append((run.get("timestamp", ""), p))
    return max(found)[1] if found else None

def run_eval(suite, extra="", compare=None):
    """One suite (router | qa_text | qa_image | drawing | e2e) against the model in memory. Rerun freely.
    compare: None = latest pinned run of the same model; a results file (path, or a name found under eval/results/);
    False = no comparison."""
    if compare is None:
        base = baseline(suite)
    elif compare is False:
        base = None
    else:
        base = Path(compare)
        if not base.is_file():                # a bare file name: look it up under eval/results/
            hits = sorted(Path("eval/results").rglob(base.name))
            if not hits:
                raise FileNotFoundError(f"no results file {compare!r} under eval/results/")
            base = hits[0]
        run = json.loads(base.read_text())["run"]
        if run.get("suite") != suite:
            raise ValueError(f"{base} is a {run.get('suite')!r} run, not {suite!r}")
        if run.get("model") != MODEL:
            print(f"note: {base.name} used {run.get('model')}, this run uses {MODEL}")
    print(f"{suite}: baseline {base or ('none' if compare is False else f'none for {MODEL}')}")
    argv = [suite, "--config", str(EVAL_CONFIG), "--split", "all", "--fresh", "--label", LABEL, *extra.split()]
    if base:
        argv += ["--compare", str(base)]
    try:
        eval_main(argv)
    except SystemExit as e:                   # argparse errors / --help: report, don't kill the kernel's state
        if e.code not in (0, None):
            print(f"eval stopped: {e}")

evaluating configs/colab_vllm_awq.yaml (qwen2.5-vl-7b-awq) as 'awq-l4'


In [19]:
# Rerun freely, with this or another suite: router | qa_text | qa_image | drawing | e2e
# (extra="--n 5" for a quick sample). The chat app can stay up; chats during a run only skew its latency numbers.
# Compare with a specific run: compare="20261019-132202_qa_text_awq-l4.json" (or a path); compare=False: none.

run_eval("qa_image", compare="20261021-163539_qa_image_bf16-l4.json")

note: 20261021-163539_qa_image_bf16-l4.json used qwen2.5-vl-7b, this run uses qwen2.5-vl-7b-awq
qa_image: baseline eval/results/20261021-163539_qa_image_bf16-l4.json



id                     split input      kind           pass  doc  fig  facts  acts  filler  problem
img-24-013-f2          dev   image+text figure         ✗     ✗    ✗    ✓      ✓             
img-24-013-f3          test  image      figure         ✗     ✗    ✗    ✗      ✓             missing ['overstress']; forbidden ['corrosion']
img-24-013-f4          dev   image      figure         ✗     ✗    ✗    ✓      ✓             
img-16-006-f1          test  image      figure         ✓     ✓    ✓    ✓      ✓             
img-16-006-f3          dev   image      figure         ✓     ✓    ✓    ✓      -             
img-16-006-f7          test  image      figure         ✗     ✓    ✓    ✗      -             missing ['inspection hole']
img-16-006-f9          dev   image      figure         ✓     ✓    ✓    ✓      -             
img-16-006-f12         test  image+text figure         ✓     ✓    ✓    ✓      ✓             
img-18-061-f2          dev   image      figure         ✓     ✓    ✓    ✓      ✓  

In [20]:
run_eval("e2e", compare="20261021-165057_e2e_bf16-l4.json")

note: 20261021-165057_e2e_bf16-l4.json used qwen2.5-vl-7b, this run uses qwen2.5-vl-7b-awq
e2e: baseline eval/results/20261021-165057_e2e_bf16-l4.json



id   split ok  route  parts   what / problem
e01  dev   ✓   ✓      ✓       text only
e02  dev   ✓   ✓      ✓       text only, multi-aircraft
e03  dev   ✗   ✓      ✗       photo only  <- answer failed facts/doc_hit (missing ['overstress'], forbidden ['corrosion'])
e04  dev   ✓   ✓      ✓       photo + question
e05  dev   ✓   ✓      ✓       drawing PDF
e06  dev   ✗   ✓      ✗       drawing as a picture  <- ctc_01_sheet1_200dpi.jpg: recall 0.588 precision 0.857
e07  dev   ✓   ✓      ✓       several drawings
e08  dev   ✓   ✓      ✓       text + drawing: current behaviour = drawing JSON only, the question is not answered
e09  test  ✓   ✓      ✓       text only, multi-aircraft
e10  test  ✗   ✓      ✗       text only, must refuse  <- answer failed refusal
e11  test  ✓   ✓      ✓       photo only
e12  test  ✓   ✓      ✓       photo + question
e13  test  ✓   ✓      ✓       drawing PDF (unverified GT)
e14  test  ✓   ✓      ✓       drawing PDF, 2 sheets (unverified GT)
e15  test  ✓   ✓      ✓   

In [21]:
run_eval("qa_text", compare="20261021-171532_qa_text_bf16-l4.json")

note: 20261021-171532_qa_text_bf16-l4.json used qwen2.5-vl-7b, this run uses qwen2.5-vl-7b-awq
qa_text: baseline eval/results/20261021-171532_qa_text_bf16-l4.json



id    split pass  facts  pages  filler   secs  problem
q01   dev   ✓     ✓      ✓                  4  
q02   dev   ✗     ✗      ✓                  4  missing ['N545PZ']
q03   dev   ✓     ✓      ✓                  4  
q04   dev   ✓     ✓      ✓                279  
q05   dev   ✗     ✗      ✓                  4  missing ['10']
q06   dev   ✓     ✓      ✓                  5  
q07   dev   ✓     ✓      ✓                  4  
q08   dev   ✓     ✓      ✓                  6  
q09   dev   ✓     ✓      ✓                  7  
q10   dev   ✓     ✓      ✓                  6  
q11   dev   ✓     ✓      ✓                  6  
q12   dev   ✓     ✓      ✓                  4  
q13   dev   ✓     ✓      ✓                  4  
q14   dev   ✗     ✗      ✓                  6  missing ['RD-1']
q15   dev   ✓     ✓      ✓                  5  
q16   dev   ✗     ✗      ✓                  4  missing ['bent', 'corrosion', 'roll groove']
q17   dev   ✓     ✓      ✓                  3  
q18   dev   ✗     ✗      ✓          

## Reading saved runs back

With `"none"` each run is the downloaded `docagent-run-<RUN_ID>.zip`. On Drive they are in
`MyDrive/ColabProjects/run_out/<RUN_ID>/`. On Hugging Face, fetch any earlier run:
```python
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} --repo-type dataset --include "runs/<RUN_ID>/*" --local-dir /content/saved
```